# KWS Training

## Import the required modules

In [1]:
import os
import random
import numpy as np
import torch
import torchaudio
import torchaudio.transforms as T
from torch import nn
from torch.utils.data import Dataset
from torch.nn import Module
from plinio.cost import params
from plinio.methods import PIT
import onnxruntime as ort
from onnxruntime.quantization import (
    CalibrationDataReader,
    CalibrationMethod,
    QuantFormat,
    QuantType,
    StaticQuantConfig,
    quantize,
)

## MSCDataset class definition

In [2]:
class MSCDataset(Dataset):
	def __init__(self, path: str, classes: list[str], transform: Module = None):
		self.labels = []
		self.data = []
		self.sampling_rates = []
		self.classes = classes
		self.transform = transform

		# check if the dataset directory exists
		if not os.path.exists(path):
			raise FileNotFoundError()

		# check if dataset path is a directory
		if not os.path.isdir(path):
			raise NotADirectoryError()
		
		# get files from path directory
		contents = os.listdir(path)
		
		for content in contents:
			# check if the current file is a .wav
			name, ext = os.path.splitext(content)
			if ext == ".wav":
	   
				# extract label from filenam
				label = name.split("_")[0]

				# add to dataset if it is related to a class in the list
				if label in self.classes:
					# Read class id, audio data and sampling rate and store them
					id = self.label_to_int(label)
					x, sampling_rate = torchaudio.load(path + content)
					target_len = 16_000

					# If the waveform is shorter than target_len → pad with zeros
					if x.shape[1] < target_len:
						pad_len = target_len - x.shape[1]
						x = torch.nn.functional.pad(x, (0, pad_len))

					# If it's longer → you may crop (optional)
					elif x.shape[1] > target_len:
						x = x[:, :target_len]

					# Apply transform
					d = self.transform(x)

					self.data.append(d)
					self.labels.append(id)
					self.sampling_rates.append(sampling_rate)

	 
	def __len__(self) -> int:
		return len(self.data)

	def __getitem__(self, idx) -> dict:
		return {
			"x": self.data[idx],
			"sampling_rate": self.sampling_rates[idx],
			"label": self.labels[idx]
		}

	def label_to_int(self, str) -> int:
		return self.classes.index(str)


## Define the Hyperparameters

In [ ]:
CFG = {
    'sampling_rate': 16000,

    'frame_length_in_s': 0.016,     # 16 ms 
    'frame_step_in_s': 0.012,       # 12 ms

    # Reduced to speed up processing
    'n_mels': 12,                  
    'n_mfcc': 8,                  

    'f_min': 50,                    # low noise cut = less processing
    'f_max': 8000,

    'train_steps': 3000,           
    'train_batch_size': 32,
    'finetune_steps': 1000,

    
    'reg_strength': 2e-6, # slightly higher because input is more compact
    'seed': 0
}


## Define the target classes

In [4]:
CLASSES=["stop", "up"]

## Set Deterministic Behaviour

In [5]:
torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
random.seed(CFG['seed'])

torch.set_default_device("cpu")

## Create Datasets and Dataloaders for train/test

In [6]:
def get_mfcc_transform(cfg):
    return T.MFCC(
        sample_rate=cfg['sampling_rate'],
        n_mfcc=cfg['n_mfcc'],
        log_mels=True,
        melkwargs=dict(
            n_fft=int(cfg['frame_length_in_s'] * cfg['sampling_rate']),
            win_length=int(cfg['frame_length_in_s'] * cfg['sampling_rate']),
            hop_length=int(cfg['frame_step_in_s'] * cfg['sampling_rate']),
            center=False,
            f_min=cfg['f_min'],
            f_max=cfg['f_max'],
            n_mels=cfg['n_mels'],
        )
    )

# Transform
transform = get_mfcc_transform(CFG)
dataset_path = '.'

# Dataset e DataLoader
train_ds = MSCDataset(path=f'{dataset_path}/msc-train/', classes=CLASSES, transform=transform)
test_ds  = MSCDataset(path=f'{dataset_path}//msc-test/', classes=CLASSES, transform=transform)
val_ds   = MSCDataset(path=f'{dataset_path}//msc-val/',  classes=CLASSES, transform=transform)

sampler = torch.utils.data.RandomSampler(
    train_ds,
    replacement=True,
    num_samples=CFG['train_steps'] * CFG['train_batch_size']
)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=CFG['train_batch_size'], sampler=sampler)
test_loader  = torch.utils.data.DataLoader(test_ds, batch_size=100)

sampler = torch.utils.data.RandomSampler(
    val_ds,
    replacement=True,
    num_samples=CFG['train_steps'] * CFG['train_batch_size'],
)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=CFG['train_batch_size'], sampler=sampler, num_workers=0)



## Create the Model

In [ ]:
class AudioCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Block 1
        self.conv1 = nn.Conv2d(1, 128, kernel_size=3, stride=2, padding=0, bias=False)
        self.bn1   = nn.BatchNorm2d(128)
        self.relu1 = nn.ReLU()

        # Block 2
        self.dconv2 = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(128)
        self.relu2 = nn.ReLU()

        # Block 3
        self.sconv3 = nn.Conv2d(128, 128, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn3   = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU()

        # Block 4
        self.dconv4 = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn4 = nn.BatchNorm2d(128)
        self.relu4 = nn.ReLU()

        # Block 5
        self.sconv5 = nn.Conv2d(128, 128, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn5   = nn.BatchNorm2d(128)
        self.relu5 = nn.ReLU()

        # Global Average Pooling + 1×1 conv
        self.gap      = nn.AdaptiveAvgPool2d((1, 1))

        # Classifier
        self.flatten  = nn.Flatten()
        self.fc       = nn.Linear(128, 2)

    def forward(self, x):
        # Block 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)

        # Block 2
        x = self.dconv2(x)
        x = self.bn2(x)
        x = self.relu2(x)

        # Block 3
        x = self.sconv3(x)
        x = self.bn3(x)
        x = self.relu3(x)

        # Block 4
        x = self.dconv4(x)
        x = self.bn4(x)
        x = self.relu4(x)

        # Block 5
        x = self.sconv5(x)
        x = self.bn5(x)
        x = self.relu5(x)

        # Global Pool + 1×1 conv
        x = self.gap(x)

        # Classifier
        x = self.flatten(x)
        x = self.fc(x)
        return x


## Prepare the Model for Pruning

To make a model "optimizable", it suffices to pass it to the PliNIO optimization method constructor (`PIT` in this example). The constructor internally implements the conversion steps necessary to generate the "searchable" network sketched in the figure above, replacing all Conv. and Linear layers with the respective mask-equipped versions, and handling all the necessary shape propagations to correctly estimate the DNN cost.

The constructor takes three main parameters:
- The previously created `nn.Module` of the "seed" DNN 
- The shape of a single input sample.
- A cost estimator, that is, the metric that we want to consider as "DNN cost". In this example, it will be the number of parameters (called `params` in PLiNIO). Other alternatives include the number of FLOPS (`ops` in PLiNIO) and hardware-dependent cost models for different targets.

In addition, we  also set the optional `discrete_cost=True`, which ensures that cost is estimated using the discretized version of the masks (i.e. $\theta$, rather than $\alpha$).

In [8]:
model = AudioCNN()
input_shape = train_ds[0]['x'].numpy().shape
pit_model = PIT(model, input_shape=input_shape, cost=params, discrete_cost=True)
print(f"Estimated DNN cost: {pit_model.cost}")
print(model)

Estimated DNN cost: 329090.0
AudioCNN(
  (conv1): Conv2d(1, 128, kernel_size=(3, 3), stride=(2, 2), bias=False)
  (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu1): ReLU()
  (dconv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu2): ReLU()
  (sconv3): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu3): ReLU()
  (dconv4): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu4): ReLU()
  (sconv5): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (bn5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu5): ReLU()
  (gap): Ada

## Define the Training Loss and Optimizer
We are now ready to run the pruning optimization loop.  For this, we will use two different optimizers for the DNN weights $W$ and for the mask parameters $\theta$. Part of the reason for this is that we should not apply *weight decay* to the mask parameters. Moreover, we may also want use a different type of optimizer (e.g. SGD vs Adam) and/or learning rate for the two parameter sets.

PLiNIO offers a convenient API to get only the normal DNN weights $W$, with the `model.net_parameters()` method, or only the NAS parameters $\theta$, with `model.nas_parameters()`. We use these two assign each optimizer to the correct parameters:

In [9]:
loss_module = torch.nn.CrossEntropyLoss()
net_optimizer = torch.optim.Adam(pit_model.net_parameters(), 1.0e-3)
mask_optimizer = torch.optim.Adam(pit_model.nas_parameters(), 1.0e-3, weight_decay=0.)

## Define the Training Loop
Lastly, let's define a modified training loop for the optimization. This is **very similar to a standard PyTorch training loop**, with a few key differences:

- It uses the two separate optimizers for $W$ and $\theta$ just created.
- It uses a modified loss function (`pruning_criterion`) that combines the two loss terms $\mathcal{L_{task}}$ and $\mathcal{L_{cost}}$ using the regularization strength $\lambda$, as detailed at the beginning of the notebook.
- In each epoch, we first train the $W$ on the training set, then train the $\theta$ on the **validation** set. This ensures that we select architectures that generalize well on data not included in the training set.

In [10]:
with torch.no_grad():
    for p in pit_model.nas_parameters():
        print((torch.abs(p)>0.5).int().numpy())

for step, (train_batch, val_batch) in enumerate(zip(train_loader, val_loader)):
    pit_model.train()

    train_x = train_batch['x']
    train_labels = train_batch['label']

    train_preds = pit_model(train_x)
    train_loss = loss_module(train_preds, train_labels)
    train_loss.backward()
    net_optimizer.step()
    net_optimizer.zero_grad()

    if ((step + 1) % 100) == 0 or step == 0:
        print(f'Step={step}; Net Loss={train_loss.item():.3f}')

    if step >= int(CFG['train_steps'] * 0.2):
        val_x = val_batch['x']
        val_labels = val_batch['label']
        val_preds = pit_model(val_x)
        val_loss = loss_module(val_preds, val_labels)
        reg_loss = CFG['reg_strength'] * pit_model.cost
        nas_loss = val_loss + reg_loss
        nas_loss.backward()
        mask_optimizer.step()
        mask_optimizer.zero_grad()

        if ((step + 1) % 100) == 0 or step == 0:
            cost = pit_model.cost.item()
            print(f'Step={step}; Val Loss={val_loss.item():.3f}; Reg Loss={reg_loss.item():.3f}; NAS Loss={nas_loss.item():.3f}; Cost={cost}')


[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 

## Final Model Export and Fine-Tuning

This model is still an instance of the `PIT` class. In order to obtain a standard `nn.Module` that we can convert to ONNX for deployment, we can leverage  the `model.export()` API of PLiNIO.

In [11]:
with torch.no_grad():
    for p in pit_model.nas_parameters():
        print((torch.abs(p)>0.5).int().cpu().numpy())


final_model = pit_model.export()

sampler = torch.utils.data.RandomSampler(
    train_ds,
    replacement=True,
    num_samples=CFG['finetune_steps'] * CFG['train_batch_size'],
)
finetune_loader = torch.utils.data.DataLoader(
    train_ds,
    batch_size=CFG['train_batch_size'],
    sampler=sampler,
    num_workers=0,
)

optimizer = torch.optim.Adam(final_model.parameters(), 1.0e-4)

for step, train_batch in enumerate(finetune_loader):
    final_model.train()

    train_x = train_batch['x']
    train_labels = train_batch['label']

    train_preds = final_model(train_x)
    train_loss = loss_module(train_preds, train_labels)
    train_loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if ((step + 1) % 100) == 0 or step == 0:
        print(f'Step={step}; Net Loss={train_loss.item():.3f}')

[1 0 0 1 1 1 0 0 0 1 0 0 1 0 1 1 0 1 1 0 0 0 1 0 1 1 1 1 1 0 0 1 0 1 0 1 0
 1 0 0 0 0 0 0 1 0 1 1 1 0 0 0 0 0 0 1 0 1 1 0 1 1 1 0 1 1 1 1 1 1 0 1 1 1
 1 0 1 1 1 1 1 1 1 1 1 1 0 1 0 1 1 0 0 0 0 1 0 1 1 0 0 1 0 1 0 0 1 0 0 0 0
 1 1 1 0 1 0 0 0 1 1 1 1 1 1 1 1 1]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 1 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1]
[0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 1 0 0 1 1 1 0 0 0 1 0 0 1 0
 1 1 1 0 1 1 1 0 0 0 0 1 0 0 1 1 0 0 1 1 0 1 1 0 0 0 0 0 0 0 1 0 1 0 0 0 0
 0 0 0 0 0 1 0 0 0 0 1 1 0 0 0 1 0 0 0 0 0 0 0 1 1 0 0 1 0 0 0 0 0 1 1 0 0
 0 1 0 0 0 0 0 0 1 0 0 0 1 1 1 0 1]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 

## Save pre quantization models

In [12]:
saved_model_dir = './saved_models/'
if not os.path.exists(saved_model_dir):
    os.makedirs(saved_model_dir)

MODEL_NAME = "Group5"

frontend_file = f'{saved_model_dir}/{MODEL_NAME}_frontend.onnx'
model_file_float32 = f'{saved_model_dir}/{MODEL_NAME}_premodel.onnx'

torch.onnx.export(
    transform,  # model to export
    torch.randn(1, 1, 16000),  # inputs of the model,
    frontend_file,  # filename of the ONNX model
    input_names=['input'], # input name in the ONNX model
    dynamo=True,
    optimize=True,
    report=False,
    external_data=False,
)

final_model = final_model.eval()

torch.onnx.export(
    final_model,  # model to export
    train_ds[0]['x'].unsqueeze(0),  # inputs of the model,
    model_file_float32,  # filename of the ONNX model
    input_names=['input'], # input name in the ONNX model
    dynamo=True,
    optimize=True,
    report=False,
    external_data=False,
)

W1220 11:42:36.798000 64275 site-packages/torch/onnx/_internal/exporter/_registration.py:103] torchvision is not installed. Skipping torchvision::nms


[torch.onnx] Obtain model graph for `MFCC([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MFCC([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


W1220 11:42:37.204000 64275 site-packages/torch/onnx/_internal/exporter/_registration.py:103] torchvision is not installed. Skipping torchvision::nms


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 1 of general pattern rewrite rules.
[torch.onnx] Obtain model graph for `GraphModule([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `GraphModule([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 10 of general pattern rewrite rules.


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.7.0',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[1,1,8,83]>
            ),
            outputs=(
                %"linear"<FLOAT,[1,2]>
            ),
            initializers=(
                %"conv1.weight"<FLOAT,[72,1,3,3]>{Tensor(...)},
                %"dconv2.weight"<FLOAT,[7,72,3,3]>{Tensor(...)},
                %"sconv3.weight"<FLOAT,[39,7,1,1]>{Tensor(...)},
                %"dconv4.weight"<FLOAT,[4,39,3,3]>{Tensor(...)},
                %"sconv5.weight"<FLOAT,[128,4,1,1]>{Tensor(...)},
                %"fc.weight"<FLOAT,[2,128]>{TorchTensor(...)},
                %"fc.bias"<FLOAT,[2]>{TorchTensor<FLOAT,[2]>(Parameter containing: tensor([-0.0517, -0.0585], requires_grad=True), name='fc.bias')}

## Post Quantization

In [13]:
ort_frontend = ort.InferenceSession(frontend_file)

class DataReader(CalibrationDataReader):
    def __init__(self, dataset):
        self.dataset = dataset
        self.enum_data = None

        self.datasize = len(self.dataset)

    def get_next(self):
        if self.enum_data is None:
            self.enum_data = iter(self.dataset)

        x = next(self.enum_data, None)

        if x is None:
            return None

        x = x['x']
        x = x.numpy()
        x = np.expand_dims(x, 0)
        x = ort_frontend.run(None, {'input': x})[0]
        x = {'input': x}

        return x

    def rewind(self):
        self.enum_data = None

calibration_ds = MSCDataset(f'{dataset_path}/msc-val/', classes=CLASSES, transform=torch.nn.Identity())
data_reader = DataReader(calibration_ds)

conf = StaticQuantConfig(
    calibration_data_reader=data_reader,
    quant_format=QuantFormat.QDQ,
    calibrate_method=CalibrationMethod.MinMax,
    activation_type=QuantType.QInt8,
    weight_type=QuantType.QInt8,
    per_channel=False,
)

model_int8_file = f'./{saved_model_dir}/{MODEL_NAME}_model.onnx'
quantize(model_file_float32, model_int8_file, conf)

frontend_size = os.path.getsize(frontend_file)
model_size = os.path.getsize(model_int8_file)
total_float32_size = frontend_size + model_size

print(f'Frontend Size: {frontend_size / 2**10:.1f}KB')
print(f'Model Size: {model_size / 2**10:.1f}KB')
print(f'Total Size: {total_float32_size / 2**10:.1f}KB')


Frontend Size: 194.8KB
Model Size: 23.7KB
Total Size: 218.5KB
